## Librerias

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats


## Carga datos clean

In [ ]:
df_operaciones = pd.read_csv('C:\\Users\\Usuario\\Desktop\\PROYECTO\\operaciones_09-03-2026.csv', parse_dates=['insert_date'])
df_operaciones

#### Transformación/creación de variables y df

In [ ]:
RUTA_FIGURES = Path(r"C:\Users\Usuario\Desktop\PROYECTO")

def guardar_grafico(fig, nombre_archivo, formato="png", dpi=300):
    RUTA_FIGURES.mkdir(parents=True, exist_ok=True)

    ruta_archivo = RUTA_FIGURES / f"{nombre_archivo}.{formato}"

    fig.savefig(ruta_archivo, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

    print(f"Gráfico guardado en: {ruta_archivo}")

In [ ]:
total_anuncios = df_operaciones["apartment_id"].nunique()                      
apartamentos_disponibles = df_operaciones["has_availability"].sum()  
apartamentos_disponibles                      
available_30 = (df_operaciones["availability_30"] > 0).sum()                            
ratio_oferta_disponible = round((available_30 / apartamentos_disponibles) * 100, 2)     
fully_booked_active = df_operaciones[
    (df_operaciones["fully_booked"]) & 
    (df_operaciones["has_availability"])
].shape[0]       

Para evitar que el análisis de disponibilidad se vea afectado por anuncios que no aceptan reservas, se filtró el dataset para considerar únicamente los alojamientos con disponibilidad activa (has_availability = True). De este modo, los cálculos reflejan únicamente la oferta realmente disponible en el mercado.

In [ ]:
df_activos = df_operaciones[df_operaciones["has_availability"] == True].copy()

## Análisis / visualizaciones

In [ ]:
palette = sns.color_palette("YlGnBu", 8)
cmap = "YlGnBu"

In [ ]:
ratio_oferta_activa = (
    df_operaciones["has_availability"].mean().round(2) * 100
)
ratio_oferta_activa

#### Disponibilidad media por ciudad

In [ ]:
availability_cols = [
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365"
]

disponibilidad_ciudad_raw = (
    df_activos
    .groupby("city")[availability_cols]
    .mean()
    .round(0)
    .reset_index()
    .sort_values(by="availability_30", ascending=True)
)

disponibilidad_ciudad_raw

Para poder comparar los distintos horizontes temporales (30, 60, 90 y 365 días), primero se transformaron los valores de disponibilidad en ratios relativos, dividiendo el número de días disponibles entre el total de días de cada periodo. De esta forma se obtiene el porcentaje de disponibilidad. Posteriormente se calculó la media de estos ratios por ciudad, lo que permite realizar comparaciones más justas entre ciudades y horizontes temporales.

In [ ]:
df_activos["avail_ratio_30"] = df_activos["availability_30"] / 30
df_activos["avail_ratio_60"] = df_activos["availability_60"] / 60
df_activos["avail_ratio_90"] = df_activos["availability_90"] / 90
df_activos["avail_ratio_365"] = df_activos["availability_365"] / 365

avail_ratio_cols = [
    "avail_ratio_30",
    "avail_ratio_60",
    "avail_ratio_90",
    "avail_ratio_365"
]

disponibilidad_ciudad_pct = (
    df_activos
    .groupby("city")[avail_ratio_cols]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)

disponibilidad_ciudad_pct

#### Oferta por ciudad

Para contextualizar la disponibilidad también se analiza
la oferta total de alojamientos por ciudad.

Esto permite interpretar mejor los resultados, ya que una ciudad
puede mostrar baja disponibilidad simplemente porque tiene
pocos alojamientos.

In [ ]:
df_ciudad_oferta = (
    df_activos
    .groupby("city")
    .agg(
        oferta_total=("apartment_id", "nunique"),
        oferta_disponible=("availability_30", lambda x: (x > 0).sum()),
        disponibilidad_media=("avail_ratio_30", "mean")
    )
    .reset_index()
)

df_ciudad_oferta["disponibilidad_media"] = (df_ciudad_oferta["disponibilidad_media"] * 100).round(2)
df_ciudad_oferta["ratio_oferta_disponible"] = (
    df_ciudad_oferta["oferta_disponible"] /
    df_ciudad_oferta["oferta_total"] * 100).round(2)
df_ciudad_oferta["ocupacion_media"] = (
    100 - df_ciudad_oferta["disponibilidad_media"]
).round(2)

df_ciudad_oferta

In [ ]:
periodos = {
    "30d": ("availability_30", 30),
    "60d": ("availability_60", 60),
    "90d": ("availability_90", 90),
    "365d": ("availability_365", 365)
}
kpi_operaciones = {
    "anuncios_totales": int(total_anuncios)
}

for p, (col, dias) in periodos.items():
    
    disponibles = (df_operaciones[col] > 0).sum()
    
    ocupacion = ((dias - df_operaciones[col]) / dias).mean() * 100
    
    kpi_operaciones[f"disponibles_{p}"] = int(disponibles)
    kpi_operaciones[f"ocupacion_{p}_%"] = round(ocupacion,1)

kpi_operaciones_df = pd.DataFrame(
    list(kpi_operaciones.items()),
    columns=["KPI", "Valor"]
)

kpi_operaciones_df

### TEST ANOVA

H0
Todas las ciudades tienen la misma disponibilidad media.

H1 
Al menos una ciudad tiene una media diferente.

Para comprobar si las diferencias observadas en la disponibilidad entre ciudades eran estadísticamente significativas, se realizó un análisis de varianza (ANOVA). El resultado mostró un valor F = 16.80 y un p-value extremadamente bajo (p < 0.001), lo que permite rechazar la hipótesis nula de igualdad de medias. Esto indica que existen diferencias significativas en la disponibilidad media de alojamientos entre las distintas ciudades analizadas.

In [ ]:
grupos = [
    grupo["avail_ratio_30"].values
    for name, grupo in df_activos.groupby("city")
]

anova = stats.f_oneway(*grupos)

print("F-statistic:", anova.statistic)
print("p-value:", anova.pvalue)

## Resultados / archivos generados

Para contextualizar el análisis de disponibilidad se analizó también la diferencia entre oferta total y oferta activa por ciudad. Esto permite entender qué proporción del mercado está realmente disponible para reservas y evita interpretar las diferencias de disponibilidad únicamente como resultado de la demanda.

In [ ]:
df_pipeline = (
    df_operaciones
    .groupby("city")
    .agg(
        oferta_total=("apartment_id", "nunique"),
        oferta_activa=("has_availability", "sum"),
        oferta_disponible_30=("availability_30", lambda x: (x > 0).sum())
    )
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(10,6))

x = np.arange(len(df_pipeline["city"]))
width = 0.25

# barras de oferta
ax1.bar(df_ciudad_oferta["city"], df_ciudad_oferta["oferta_total"], color="lightblue")
ax1.set_ylabel("Número de alojamientos")

# segundo eje para ocupación
ax2 = ax1.twinx()
ax2.plot(
    df_ciudad_oferta["city"],
    df_ciudad_oferta["ocupacion_media"],
    marker="o",
    color="darkgreen"
)

for x, y in zip(df_ciudad_oferta["city"], df_ciudad_oferta["ocupacion_media"]):
    ax2.text(x, y + 0.3, f"{y:.1f}%", ha='center')


ax1.set_xticklabels(df_pipeline["city"])

ax1.set_ylabel("Número de alojamientos")
ax1.set_xlabel("Ciudad")
ax1.set_title("Tamaño del mercado y ocupación media por ciudad (mensual)")
ax2.set_ylabel("Ocupación media (%)")


plt.show()

In [ ]:
data = disponibilidad_ciudad_pct.set_index("city")
labels = data.round(2).astype(str) + "%"

fig, ax = plt.subplots(figsize=(10,6))

sns.heatmap(
    data,
    annot=labels,
    cmap=cmap,
    fmt=""
)

ax.set_title("Disponibilidad relativa por ciudad")
ax.set_ylabel("Ciudad")
ax.set_xlabel("Disponibilidad temporal")
ax.set_xticklabels(["Mensual","Bimensual","Trimestral","Anual"])

plt.show()

In [ ]:
orden = df_ciudad_oferta.sort_values("disponibilidad_media")["city"]
df_activos["avail_ratio_30_pct"] = df_activos["avail_ratio_30"] * 100

fig, ax = plt.subplots(figsize=(12,6))
sns.boxplot(
    data=df_activos,
    x="city",
    y="avail_ratio_30_pct",
    order=orden,
    palette="YlGnBu"
)
for i, city in enumerate(orden):

    oferta = df_ciudad_oferta.loc[
        df_ciudad_oferta["city"] == city,
        "oferta_total"
    ].values[0]

    ax.text(i, 101, f"N={oferta}", ha="center")

ax.set_ylabel("Disponibilidad (%)")
ax.set_xlabel("Ciudad")
ax.set_title("Distribución de disponibilidad por ciudad\n(con tamaño de oferta)")

plt.show()

Conclusiones: La disponibilidad de alojamientos aumenta conforme el horizonte temporal es mayor, lo que refleja un comportamiento esperado en mercados turísticos donde las reservas se concentran en el corto plazo. Madrid y Barcelona presentan los niveles más bajos de disponibilidad mensual, lo que sugiere una mayor presión de demanda. En contraste, ciudades como Girona y Menorca muestran mayores niveles de disponibilidad relativa, lo que podría indicar menor demanda o mayor oferta de alojamientos.